# BusesToken Demo with Built-in IEEE Networks

This notebook demonstrates `pypowsybl-to-busestoken` without any private grid data.
It uses the six IEEE test networks bundled with PyPowSyBl:

- IEEE 9-bus
- IEEE 14-bus
- IEEE 30-bus
- IEEE 57-bus
- IEEE 118-bus
- IEEE 300-bus

The goal is simple: a reader can clone the repository, install the package, run this notebook, and immediately see how a solved power-grid network becomes bus tokens, branch relation features, and a sparse graph index ready for ML models.


## 1. Imports

We only use public PyPowSyBl network factories and the package converter. No external XIIDM file is required for this demo.


In [1]:
import numpy as np
import pandas as pd
import pypowsybl as pp

import pypowsybl_to_busestoken as pb
from pypowsybl_to_busestoken import BusesTokenConverter

print('pypowsybl_to_busestoken exports:', pb.__all__)
print('pypowsybl version:', getattr(pp, '__version__', 'unknown'))

pypowsybl_to_busestoken exports: ['BusesTokenConverter', 'BusesToken', 'BusesTokenScaler', 'make_component_mode_kwarg', 'ready_to_use']
pypowsybl version: 1.14.0


## 2. Define the IEEE Cases

Each factory returns a PyPowSyBl `Network`. The converter will run an AC load flow, keep the solved active buses, extract active line/transformer relations, and build a `BusesToken` object.


In [2]:
IEEE_CASES = {
    'ieee9': pp.network.create_ieee9,
    'ieee14': pp.network.create_ieee14,
    'ieee30': pp.network.create_ieee30,
    'ieee57': pp.network.create_ieee57,
    'ieee118': pp.network.create_ieee118,
    'ieee300': pp.network.create_ieee300,
}

converter = BusesTokenConverter(run_lf=True, provider='OpenLoadFlow')
list(IEEE_CASES)

['ieee9', 'ieee14', 'ieee30', 'ieee57', 'ieee118', 'ieee300']

## 3. Convert All IEEE Networks

For each case we record both the raw network size and the resulting token graph size. The token graph may differ from declared topology because the converter keeps the solved operational state: active buses and active branch terminals only.


In [3]:
tokens = {}
summary_rows = []

for case_name, factory in IEEE_CASES.items():
    network = factory()
    token = converter(network, snapshot_id=case_name)
    tokens[case_name] = token

    rho = token.relation_df['base_rho'].dropna()
    summary_rows.append({
        'case': case_name,
        'raw_buses': len(network.get_buses()),
        'raw_lines': len(network.get_lines()),
        'raw_2wt': len(network.get_2_windings_transformers()),
        'bus_tokens': token.n_tokens,
        'branch_relations': token.n_relations,
        'bus_features': len(token.token_feature_names),
        'relation_features': len(token.relation_feature_names),
        'relations_with_limit': int(rho.shape[0]),
        'max_base_rho': float(rho.max()) if len(rho) else np.nan,
        'mean_base_rho': float(rho.mean()) if len(rho) else np.nan,
    })

summary = pd.DataFrame(summary_rows)
summary

,case,raw_buses,raw_lines,raw_2wt,bus_tokens,branch_relations,bus_features,relation_features,relations_with_limit,max_base_rho,mean_base_rho
0,ieee9,9,6,3,9,9,17,20,0,NaN,NaN
1,ieee14,14,17,3,14,20,17,20,0,NaN,NaN
2,ieee30,30,37,4,30,41,17,20,0,NaN,NaN
3,ieee57,57,63,17,57,80,17,20,0,NaN,NaN
4,ieee118,118,177,9,118,186,17,20,0,NaN,NaN
5,ieee300,300,304,107,300,411,17,20,0,NaN,NaN


## 4. Validate Token Invariants

These checks are intentionally close to what downstream ML code expects:

- bus IDs are unique;
- branch IDs are unique;
- every relation endpoint maps to an existing bus token;
- feature arrays are finite `float32` matrices after NaN filling;
- `relation_index` is a valid sparse graph index of shape `(2, E)`.


In [4]:
def validate_token(token):
    active_buses = set(token.bus_df.index)
    relation_index = token.relation_index
    x_bus = token.token_features
    edge_attr = token.relation_features

    checks = {
        'bus_index_unique': bool(token.bus_df.index.is_unique),
        'relation_index_unique': bool(token.relation_df.index.is_unique),
        'relations_point_to_known_buses': bool(
            token.relation_df['bus1_id'].isin(active_buses).all()
            and token.relation_df['bus2_id'].isin(active_buses).all()
        ),
        'x_bus_shape_ok': x_bus.shape == (token.n_tokens, len(token.token_feature_names)),
        'edge_attr_shape_ok': edge_attr.shape == (token.n_relations, len(token.relation_feature_names)),
        'relation_index_shape_ok': relation_index.shape == (2, token.n_relations),
        'relation_index_in_bounds': bool((relation_index >= 0).all() and (relation_index < token.n_tokens).all()),
        'x_bus_no_nan': bool(not np.isnan(x_bus).any()),
        'edge_attr_no_nan': bool(not np.isnan(edge_attr).any()),
    }
    checks['all_ok'] = all(checks.values())
    return checks

validation = pd.DataFrame([
    {'case': case_name, **validate_token(token)}
    for case_name, token in tokens.items()
])
validation

,case,bus_index_unique,relation_index_unique,relations_point_to_known_buses,x_bus_shape_ok,edge_attr_shape_ok,relation_index_shape_ok,relation_index_in_bounds,x_bus_no_nan,edge_attr_no_nan,all_ok
0,ieee9,True,True,True,True,True,True,True,True,True,True
1,ieee14,True,True,True,True,True,True,True,True,True,True
2,ieee30,True,True,True,True,True,True,True,True,True,True
3,ieee57,True,True,True,True,True,True,True,True,True,True
4,ieee118,True,True,True,True,True,True,True,True,True,True
5,ieee300,True,True,True,True,True,True,True,True,True,True


## 5. Inspect One Case

We select IEEE 118 as a medium-to-large public example. The same object fields are available for every case.


In [5]:
CASE = 'ieee118'
token = tokens[CASE]
print(token)
print('token_features shape   :', token.token_features.shape)
print('relation_features shape:', token.relation_features.shape)
print('relation_index shape   :', token.relation_index.shape)

BusesToken(snapshot='ieee118', n_tokens=118, n_relations=186, token_features=17, relation_features=20)
token_features shape   : (118, 17)
relation_features shape: (186, 20)
relation_index shape   : (2, 186)


## 6. Bus Feature Schema

Each active bus becomes one token. The numerical columns are ordered and form the `token_features` matrix.


In [6]:
pd.DataFrame({'bus_feature': token.token_feature_names})

,bus_feature
0,v_mag
1,v_angle
2,nominal_v
3,is_main_component
4,gen_p
5,gen_q
6,load_p
7,load_q
8,shunt_q
9,bat_p


## 7. Bus Token Table

The bus table remains a pandas DataFrame, which makes the representation auditable before feeding it to ML models.


In [7]:
bus_preview_cols = [
    'v_mag', 'v_angle', 'nominal_v', 'is_main_component',
    'gen_p', 'load_p', 'p_net', 'q_net',
    'n_gens', 'n_loads', 'n_shunts', 'n_batteries',
]
token.bus_df[bus_preview_cols].head(10)

,v_mag,v_angle,nominal_v,is_main_component,gen_p,load_p,p_net,q_net,n_gens,n_loads,n_shunts,n_batteries
bus_id,,,,,,,,,,,,
VL1_0,131.790000,-19.092517,138.0,1.0,-0.000000,51.0,-51.000000,-30.104111,1,1,0,0
VL2_0,134.052200,-18.552480,138.0,1.0,0.000000,20.0,-20.000000,-9.000000,0,1,0,0
VL3_0,133.541490,-18.209165,138.0,1.0,0.000000,39.0,-39.000000,-10.000000,0,1,0,0
VL4_0,137.724000,-14.492022,138.0,1.0,-9.074628,30.0,-39.074628,-26.993514,1,1,0,0
VL5_0,138.273889,-14.046712,138.0,1.0,0.000000,0.0,0.000000,-40.158934,0,0,1,0
VL6_0,136.620000,-16.773524,138.0,1.0,-0.000000,52.0,-52.000000,-6.071055,1,1,0,0
VL7_0,136.527244,-17.217867,138.0,1.0,0.000000,19.0,-19.000000,-2.000000,0,1,0,0
VL8_0,350.175000,-9.025317,345.0,1.0,-28.074628,0.0,-28.074628,62.748022,1,0,0,0
VL9_0,359.807810,-1.772458,345.0,1.0,0.000000,0.0,0.000000,0.000000,0,0,0,0


## 8. Branch Relation Feature Schema

Each active line or two-winding transformer becomes a relation between two bus tokens. These relation features condition sparse message passing or sparse attention.


In [8]:
pd.DataFrame({'relation_feature': token.relation_feature_names})

,relation_feature
0,r
1,x
2,g1
3,b1
4,g2
5,b2
6,tap_rho
7,tap_alpha
8,p1
9,q1


## 9. Branch Relation Table

`bus1_id` and `bus2_id` define the physical endpoints. Electrical parameters, operating flows, current limits, and `base_rho` remain visible for inspection.


In [9]:
relation_preview_cols = [
    'bus1_id', 'bus2_id', 'branch_kind',
    'r', 'x', 'tap_rho', 'tap_alpha',
    'p1', 'q1', 'i1', 'p2', 'q2', 'i2',
    'limit1', 'limit2', 'base_rho',
    'is_line', 'is_2wt', 'is_self_loop',
]
token.relation_df[relation_preview_cols].head(10)

,bus1_id,bus2_id,branch_kind,r,x,tap_rho,tap_alpha,p1,q1,i1,p2,q2,i2,limit1,limit2,base_rho,is_line,is_2wt,is_self_loop
branch_id,,,,,,,,,,,,,,,,,,,
L1-2-1,VL1_0,VL2_0,LINE,5.770332,19.024956,1.0,0.0,-12.356194,-13.040101,78.699090,12.453821,11.005329,71.579555,NaN,NaN,NaN,1,0,0
L1-3-1,VL1_0,VL3_0,LINE,2.456676,8.074656,1.0,0.0,-38.643806,-17.064009,185.062373,38.893868,16.885905,183.316647,NaN,NaN,NaN,1,0,0
L4-5-1,VL4_0,VL5_0,LINE,0.335174,1.519711,1.0,0.0,-103.275914,-26.782181,447.261917,103.476963,27.483760,447.038875,NaN,NaN,NaN,1,0,0
L3-5-1,VL3_0,VL5_0,LINE,4.589604,20.567520,1.0,0.0,-68.102674,-14.491284,301.025429,69.340889,17.284764,298.386252,NaN,NaN,NaN,1,0,0
L5-6-1,VL5_0,VL6_0,LINE,2.266236,10.283760,1.0,0.0,88.455173,4.109169,369.734980,-87.525004,-1.302881,369.917897,NaN,NaN,NaN,1,0,0
L6-7-1,VL6_0,VL7_0,LINE,0.874120,3.961152,1.0,0.0,35.525004,-4.768174,151.473386,-35.464953,4.501611,151.178536,NaN,NaN,NaN,1,0,0
L8-9-1,VL8_0,VL9_0,LINE,2.904210,36.302625,1.0,0.0,-440.563426,-89.758728,741.300127,445.181605,24.435416,715.416952,NaN,NaN,NaN,1,0,0
L9-10-1,VL9_0,VL10_0,LINE,3.070845,38.326050,1.0,0.0,-445.181605,-24.435416,715.416952,449.925372,-51.055773,721.688492,NaN,NaN,NaN,1,0,0
L4-11-1,VL4_0,VL11_0,LINE,3.980196,13.102272,1.0,0.0,64.201286,-0.211332,269.138500,-63.336282,1.340174,269.050933,NaN,NaN,NaN,1,0,0


## 10. Sparse Relation Index

`relation_index` maps branch endpoints from bus IDs to integer token positions. It is the sparse graph structure used by graph neural networks and sparse attention blocks.


In [10]:
ri = token.relation_index
edge_lookup = pd.DataFrame({
    'src_idx': ri[0, :10],
    'dst_idx': ri[1, :10],
    'branch_id': token.relation_df.index[:10],
    'src_bus_id': token.relation_df['bus1_id'].values[:10],
    'dst_bus_id': token.relation_df['bus2_id'].values[:10],
    'branch_kind': token.relation_df['branch_kind'].values[:10],
})
edge_lookup

,src_idx,dst_idx,branch_id,src_bus_id,dst_bus_id,branch_kind
0,0,1,L1-2-1,VL1_0,VL2_0,LINE
1,0,2,L1-3-1,VL1_0,VL3_0,LINE
2,3,4,L4-5-1,VL4_0,VL5_0,LINE
3,2,4,L3-5-1,VL3_0,VL5_0,LINE
4,4,5,L5-6-1,VL5_0,VL6_0,LINE
5,5,6,L6-7-1,VL6_0,VL7_0,LINE
6,7,8,L8-9-1,VL8_0,VL9_0,LINE
7,8,9,L9-10-1,VL9_0,VL10_0,LINE
8,3,10,L4-11-1,VL4_0,VL11_0,LINE
9,4,10,L5-11-1,VL5_0,VL11_0,LINE


## 11. Loading Ratio Distribution

`base_rho` is the N-0 loading ratio derived from currents and permanent current limits. It is not read as a native field from the network; it is computed by the converter as `max(i1 / limit1, i2 / limit2)` when limits exist.


In [11]:
rho = token.relation_df['base_rho'].dropna()
rho.describe().to_frame(name='base_rho')

,base_rho
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


## 12. Optional NetworkX Conversion

The token graph can be converted to a `networkx.MultiDiGraph` for quick graph inspection. This step is optional because NetworkX is an optional dependency.


In [12]:
try:
    G = token.to_networkx()
    degree = pd.Series(dict(G.degree()), name='degree').sort_values(ascending=False)
    print(f'NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    display(degree.head(10).to_frame())
except ImportError as exc:
    print('NetworkX is not installed. Install with: pip install networkx')

NetworkX graph: 118 nodes, 186 edges


,degree
VL49_0,12
VL100_0,8
VL80_0,8
VL59_0,7
VL77_0,7
VL12_0,7
VL92_0,7
VL17_0,6
VL37_0,6
VL56_0,6


## 13. What This Demonstrates

This public demo validates the package on a range of built-in networks without requiring private data:

```text
PyPowSyBl IEEE Network
        |
        | AC load flow
        v
Active solved buses + active branch terminals
        |
        v
BusesToken(bus_df, relation_df, relation_index)
        |
        v
ML-ready matrices: x_bus, edge_attr, edge_index
```

For private or operator-specific studies, the same converter can be applied to any PyPowSyBl-supported network file, but those files should remain outside the public repository.
